In [8]:
!pip install lightgbm catboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.8/28.8 MB 31.0 MB/s eta 0:00:00m eta 0:00:010:00:01


# Test1

타깃을 "오늘 다른 종목들과 비교한 상위 30%" 대신 **"이 종목 자신의 과거 분포 대비 향후 공매도가 비정상적으로 집중될지"**로 바꾸고, 종목코드 + 날짜를 입력하면 확률(0~1)을 조회할 수 있는 형태로 코드를 수정하면 될까요?
이렇게 하면 대형주든 소형주든 "이 종목 입장에서 평소보다 공매도가 몰릴 위험이 큰가"를 동일한 기준으로 평가할 수 있어서, "이 기업 내일 어떻게 되나" 질문에 직접 답이 됩니다.


데이터셋 shape를 직접 확인해드리겠습니다.**결과 평가:**

- AUC 0.848은 "이 종목 자신의 평소 수준과 비교"라는 더 어렵고 현실적인 문제에서 0.84 이상이면 강한 신호력입니다 (참고로 횡단면 방식은 0.75였습니다).
- F1 0.758은 target 비율이 38~43%로 어느 정도 균형 잡혀 있어 신뢰할 만한 수치입니다.
- LightGBM이 LR보다 AUC +0.029, F1 +0.032로 일관되게 우위 — "비선형 모델의 우위"라는 발표 메시지에 정확히 부합합니다.
- Train(0.377) vs Test(0.428) target 비율 차이는 자연스러운 수준(시장 전체 공매도 활동이 후반기에 다소 증가했을 가능성)이고, 모델이 이 변화에도 잘 적응한 것으로 보입니다.

**데이터셋 shape:**

| 단계 | shape | 설명 |
|---|---|---|
| 원본 병합 (load_data) | (248,125, 15) | price + short_merged, 948개 종목 × 263거래일 (2025-03-04~2026-03-31) |
| 피처/타깃 추가 후 | (248,125, 47) | 26개 피처 + 타깃 관련 컬럼 |
| Train (dropna 후) | (152,816, 47) | 901종목 × 193일 |
| Val | (15,971, 47) | 887종목 × 20일 |
| Test | (16,489, 47) | 907종목 × 20일 |
| 학습 가능 행 합계 | 185,276 / 248,125 (74.7%) | rolling/expanding window로 초기 구간(MIN_HIST_DAYS=30일, 20일 이동평균 등) 소실 |


In [1]:
"""
XAI 기반 공매도 집중 신호 탐지 v4
변경사항:
  - [신규] Linear Regression 추가 (선형 모델 기준선 강화)
  - [개선] 비선형 모델(LightGBM/XGBoost/CatBoost) F1 향상
           · val set 기준 최적 임계값 탐색 (F1 최대화)
           · 클래스 불균형 자동 보정 (scale_pos_weight / class_weight)
           · n_estimators·num_leaves·depth 확대

- 타깃: "이 종목 자신의 과거 분포 대비, 향후 5거래일 내 공매도가
         비정상적으로 집중될 확률" (own-history quantile 기반)
- 모델: Linear Regression · Logistic Regression ·
        LightGBM · XGBoost · CatBoost · LSTM
- 조회: predict_for(model, df, "005930", "2026-03-25")
        predict_history(model, df, "005930")
"""

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import shap

RANDOM_STATE = 42
HORIZON_DAYS = 5
VAL_SIZE_DAYS = 20
TEST_SIZE_DAYS = 20
Q = 0.7
MIN_HIST_DAYS = 30
LOOKBACK = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ------------------------------------------------------------------
# 공통 유틸: val set 기준 F1 최적 임계값 탐색
# ------------------------------------------------------------------
def find_best_threshold(y_val, proba_val, lo=0.2, hi=0.8, step=0.01):
    """val F1 를 최대화하는 임계값 반환."""
    best_thresh, best_f1 = 0.5, -1.0
    for t in np.arange(lo, hi, step):
        f = f1_score(y_val, proba_val >= t, zero_division=0)
        if f > best_f1:
            best_f1, best_thresh = f, t
    return round(float(best_thresh), 3)


# ------------------------------------------------------------------
# 1. 데이터 로드 & 병합
# ------------------------------------------------------------------
def clean_numeric(s):
    return pd.to_numeric(s.astype(str).str.replace(",", ""), errors="coerce")


def load_data():
    price = pd.read_csv("./dataset/df_price_20250401_20260331_with_60d_buffer.csv")
    merged = pd.read_csv("./dataset/df_short_merged_20250304_20260331.csv")

    for d in (price, merged):
        d["ISU_CD"] = d["ISU_CD"].astype(str).str.zfill(6)
        d["date"] = pd.to_datetime(d["date"])

    num_cols_price = ["TDD_OPNPRC", "TDD_HGPRC", "TDD_LWPRC", "TDD_CLSPRC",
                      "ACC_TRDVOL", "CHG_RT"]
    num_cols_short = ["short_qty", "acc_trdvol", "short_ratio",
                      "balance_ratio", "MKTCAP", "LIST_SHRS"]
    price[num_cols_price] = price[num_cols_price].apply(clean_numeric)
    merged[num_cols_short] = merged[num_cols_short].apply(clean_numeric)

    df = price.merge(
        merged[["ISU_CD", "date", "short_qty", "short_ratio",
                "balance_ratio", "MKTCAP", "LIST_SHRS", "acc_trdvol"]],
        on=["ISU_CD", "date"], how="inner",
    )
    return df.sort_values(["ISU_CD", "date"]).reset_index(drop=True)


# ------------------------------------------------------------------
# 2. 피처 엔지니어링
# ------------------------------------------------------------------
def add_features(df):
    g = df.groupby("ISU_CD")

    df["log_mktcap"]   = np.log(df["MKTCAP"].clip(lower=1))
    df["log_price"]    = np.log(df["TDD_CLSPRC"].clip(lower=1))
    df["is_large_cap"] = (df["MKTCAP"] >= g["MKTCAP"].transform("median")).astype(int)

    df["ret_1d"]  = g["TDD_CLSPRC"].pct_change(1)
    df["ret_5d"]  = g["TDD_CLSPRC"].pct_change(5)
    df["ret_20d"] = g["TDD_CLSPRC"].pct_change(20)
    df["vol_5d"]  = g["ret_1d"].transform(lambda x: x.rolling(5).std())
    df["vol_20d"] = g["ret_1d"].transform(lambda x: x.rolling(20).std())
    df["intraday_vol"]   = (df["TDD_HGPRC"] - df["TDD_LWPRC"]) / df["TDD_CLSPRC"]
    ma20 = g["TDD_CLSPRC"].transform(lambda x: x.rolling(20).mean())
    df["price_ma20_gap"] = (df["TDD_CLSPRC"] - ma20) / ma20

    df["turnover"]   = df["ACC_TRDVOL"] / df["LIST_SHRS"]
    trdval = df["ACC_TRDVOL"] * df["TDD_CLSPRC"]
    df["log_trdval"] = np.log(trdval.clip(lower=1))
    vol_avg20 = g["ACC_TRDVOL"].transform(lambda x: x.rolling(20).mean())
    df["abnormal_vol"] = df["ACC_TRDVOL"] / vol_avg20
    trdval_ma5  = trdval.groupby(df["ISU_CD"]).transform(lambda x: x.rolling(5).mean())
    trdval_ma20 = trdval.groupby(df["ISU_CD"]).transform(lambda x: x.rolling(20).mean())
    df["trdval_ma5_over_ma20"]  = trdval_ma5 / trdval_ma20
    df["ret5_x_abnormal_vol"]   = df["ret_5d"] * df["abnormal_vol"]

    df["short_ratio_ma5"]     = g["short_ratio"].transform(lambda x: x.rolling(5).mean())
    df["short_ratio_ma10"]    = g["short_ratio"].transform(lambda x: x.rolling(10).mean())
    df["short_ratio_lag1"]    = g["short_ratio"].shift(1)
    df["short_ratio_chg_1d"]  = g["short_ratio"].diff(1)
    df["short_ratio_std5"]    = g["short_ratio"].transform(lambda x: x.rolling(5).std())
    df["balance_ratio_chg_5d"] = g["balance_ratio"].diff(5)
    df["balance_ratio_ma5"]   = g["balance_ratio"].transform(lambda x: x.rolling(5).mean())
    df["balance_ratio_lag1"]  = g["balance_ratio"].shift(1)
    df["short_qty_chg5"]      = g["short_qty"].pct_change(5).replace([np.inf, -np.inf], np.nan)

    df["short_ratio_pct_vs_own_hist"] = g["short_ratio"].transform(
        lambda x: x.shift(1).expanding(min_periods=MIN_HIST_DAYS).rank(pct=True)
    )
    df["balance_ratio_pct_vs_own_hist"] = g["balance_ratio"].transform(
        lambda x: x.shift(1).expanding(min_periods=MIN_HIST_DAYS).rank(pct=True)
    )
    return df


FEATURE_COLS = [
    "log_mktcap", "log_price", "is_large_cap",
    "ret_1d", "ret_5d", "ret_20d", "vol_5d", "vol_20d", "intraday_vol", "price_ma20_gap",
    "turnover", "log_trdval", "abnormal_vol", "trdval_ma5_over_ma20", "ret5_x_abnormal_vol",
    "short_ratio_ma5", "short_ratio_ma10", "short_ratio_lag1",
    "short_ratio_chg_1d", "short_ratio_std5",
    "balance_ratio_chg_5d", "balance_ratio_ma5", "balance_ratio_lag1", "short_qty_chg5",
    "short_ratio_pct_vs_own_hist", "balance_ratio_pct_vs_own_hist",
]


# ------------------------------------------------------------------
# 3. 타깃 정의
# ------------------------------------------------------------------
def add_target(df):
    g = df.groupby("ISU_CD")

    df["future_short_ratio_median_h5"] = (
        g["short_ratio"].transform(lambda x: x.shift(-1).rolling(HORIZON_DAYS).median())
    )
    df["future_balance_ratio_h5"] = g["balance_ratio"].shift(-HORIZON_DAYS)
    df["future_ret_h5"] = g["TDD_CLSPRC"].shift(-HORIZON_DAYS) / df["TDD_CLSPRC"] - 1

    df["target_score_h5"] = (
        0.7 * df["future_short_ratio_median_h5"] + 0.3 * df["future_balance_ratio_h5"]
    )

    df["own_hist_q70"] = df.groupby("ISU_CD")["target_score_h5"].transform(
        lambda x: x.shift(1).expanding(min_periods=MIN_HIST_DAYS).quantile(Q)
    )
    df["target"] = (
        (df["target_score_h5"] >= df["own_hist_q70"]) & (df["target_score_h5"] > 0)
    ).astype(int)
    return df


# ------------------------------------------------------------------
# 4. 시계열 기반 분할
# ------------------------------------------------------------------
def time_split(df):
    df = df.dropna(subset=FEATURE_COLS + ["target"]).copy()
    unique_dates = sorted(df["date"].unique())

    test_dates  = unique_dates[-TEST_SIZE_DAYS:]
    val_dates   = unique_dates[-(TEST_SIZE_DAYS + VAL_SIZE_DAYS):-TEST_SIZE_DAYS]
    train_dates = unique_dates[: -(TEST_SIZE_DAYS + VAL_SIZE_DAYS)]

    train = df[df["date"].isin(train_dates)]
    val   = df[df["date"].isin(val_dates)]
    test  = df[df["date"].isin(test_dates)]
    return train, val, test


# ------------------------------------------------------------------
# 5-1. 트리 / 선형 모델 학습
#   신규: LinearRegression (predict → clip(0,1) → 확률로 사용)
#   개선: 비선형 모델에 ① 불균형 보정 ② val 기준 임계값 최적화 ③ 하이퍼파라미터 강화
# ------------------------------------------------------------------
def train_tabular_models(train, val, test):
    X_train, y_train = train[FEATURE_COLS], train["target"]
    X_val,   y_val   = val[FEATURE_COLS],   val["target"]
    X_test,  y_test  = test[FEATURE_COLS],  test["target"]

    # 클래스 불균형 비율 (비선형 모델 공통 사용)
    n_neg  = (y_train == 0).sum()
    n_pos  = (y_train == 1).sum()
    pos_weight = n_neg / max(n_pos, 1)        # XGBoost·CatBoost scale_pos_weight
    class_w    = {0: 1.0, 1: pos_weight}      # LightGBM is_unbalance 대신 직접 지정

    models, results = {}, {}
    scaler_std = StandardScaler().fit(X_train)

    # ----------------------------------------------------------------
    # [1] Linear Regression (신규)
    #   회귀 출력값을 [0,1]로 클리핑 → AUC/F1 계산
    #   선형 모델이므로 임계값 최적화 없이 0.5 고정 (해석 기준선 역할)
    # ----------------------------------------------------------------
    Xtr_s = scaler_std.transform(X_train)
    Xvl_s = scaler_std.transform(X_val)
    Xte_s = scaler_std.transform(X_test)

    linreg = LinearRegression()
    linreg.fit(Xtr_s, y_train)

    proba_linreg = linreg.predict(Xte_s).clip(0, 1)  # 확률로 변환
    models["Linear Regression"] = (linreg, scaler_std)
    results["Linear Regression"] = {
        "AUC":        roc_auc_score(y_test, proba_linreg),
        "F1":         f1_score(y_test, proba_linreg >= 0.5, zero_division=0),
        "best_thresh": 0.5,   # 선형 모델: 고정
    }

    # ----------------------------------------------------------------
    # [2] Logistic Regression (기존 유지, 임계값 최적화 추가)
    # ----------------------------------------------------------------
    lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                            class_weight="balanced")
    lr.fit(Xtr_s, y_train)

    proba_lr_val  = lr.predict_proba(Xvl_s)[:, 1]
    proba_lr_test = lr.predict_proba(Xte_s)[:, 1]
    thresh_lr = find_best_threshold(y_val, proba_lr_val)

    models["Logistic Regression"] = (lr, scaler_std)
    results["Logistic Regression"] = {
        "AUC":         roc_auc_score(y_test, proba_lr_test),
        "F1":          f1_score(y_test, proba_lr_test >= thresh_lr, zero_division=0),
        "best_thresh": thresh_lr,
    }

    # ----------------------------------------------------------------
    # [3] LightGBM — 불균형 보정 + 임계값 최적화 + 하이퍼파라미터 강화
    # ----------------------------------------------------------------
    gbm = lgb.LGBMClassifier(
        random_state=RANDOM_STATE,
        verbose=-1,
        n_estimators=1000,     # 500 → 1000
        learning_rate=0.02,    # 0.03 → 0.02
        num_leaves=127,        # 63  → 127
        min_child_samples=20,
        class_weight=class_w,  # [개선] 클래스 불균형 보정
        subsample=0.8,
        colsample_bytree=0.8,
    )
    gbm.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(80, verbose=False)],
    )
    proba_gbm_val  = gbm.predict_proba(X_val)[:, 1]
    proba_gbm_test = gbm.predict_proba(X_test)[:, 1]
    thresh_gbm = find_best_threshold(y_val, proba_gbm_val)  # [개선] 임계값 최적화

    models["LightGBM"] = (gbm, None)
    results["LightGBM"] = {
        "AUC":         roc_auc_score(y_test, proba_gbm_test),
        "F1":          f1_score(y_test, proba_gbm_test >= thresh_gbm, zero_division=0),
        "best_thresh": thresh_gbm,
    }

    # ----------------------------------------------------------------
    # [4] XGBoost — 불균형 보정 + 임계값 최적화 + 하이퍼파라미터 강화
    # ----------------------------------------------------------------
    xgbm = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.02,
        max_depth=7,           # 6 → 7
        scale_pos_weight=pos_weight,   # [개선] 클래스 불균형 보정
        random_state=RANDOM_STATE,
        eval_metric="auc",
        early_stopping_rounds=80,
        verbosity=0,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
    )
    xgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    proba_xgb_val  = xgbm.predict_proba(X_val)[:, 1]
    proba_xgb_test = xgbm.predict_proba(X_test)[:, 1]
    thresh_xgb = find_best_threshold(y_val, proba_xgb_val)  # [개선] 임계값 최적화

    models["XGBoost"] = (xgbm, None)
    results["XGBoost"] = {
        "AUC":         roc_auc_score(y_test, proba_xgb_test),
        "F1":          f1_score(y_test, proba_xgb_test >= thresh_xgb, zero_division=0),
        "best_thresh": thresh_xgb,
    }

    # ----------------------------------------------------------------
    # [5] CatBoost — 불균형 보정 + 임계값 최적화 + 하이퍼파라미터 강화
    # ----------------------------------------------------------------
    catm = cb.CatBoostClassifier(
        iterations=1000,
        learning_rate=0.02,
        depth=7,               # 6 → 7
        scale_pos_weight=pos_weight,   # [개선] 클래스 불균형 보정
        random_state=RANDOM_STATE,
        verbose=0,
        l2_leaf_reg=5.0,
        min_data_in_leaf=20,
    )
    catm.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=80)

    proba_cat_val  = catm.predict_proba(X_val)[:, 1]
    proba_cat_test = catm.predict_proba(X_test)[:, 1]
    thresh_cat = find_best_threshold(y_val, proba_cat_val)  # [개선] 임계값 최적화

    models["CatBoost"] = (catm, None)
    results["CatBoost"] = {
        "AUC":         roc_auc_score(y_test, proba_cat_test),
        "F1":          f1_score(y_test, proba_cat_test >= thresh_cat, zero_division=0),
        "best_thresh": thresh_cat,
    }

    return models, results


# ------------------------------------------------------------------
# 5-2. LSTM
# ------------------------------------------------------------------
class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


def make_sequences(df, dates_allowed):
    Xs, ys, meta = [], [], []
    for isu, g in df.groupby("ISU_CD"):
        g = g.sort_values("date").reset_index(drop=True)
        feat   = g[FEATURE_COLS].to_numpy()
        target = g["target"].to_numpy()
        valid  = ~np.isnan(feat).any(axis=1) & ~np.isnan(target)
        for i in range(LOOKBACK - 1, len(g)):
            if g["date"].iloc[i] not in dates_allowed:
                continue
            window = slice(i - LOOKBACK + 1, i + 1)
            if not valid[window].all():
                continue
            Xs.append(feat[window])
            ys.append(target[i])
            meta.append((isu, g["date"].iloc[i]))
    return np.asarray(Xs, dtype=np.float32), np.asarray(ys, dtype=np.float32), meta


def train_lstm(df, train, val, test, epochs=10, batch_size=512):
    train_dates = set(train["date"])
    val_dates   = set(val["date"])
    test_dates  = set(test["date"])

    Xtr,   ytr,   _    = make_sequences(df, train_dates)
    Xval,  yval,  _    = make_sequences(df, val_dates)
    Xtest, ytest, _    = make_sequences(df, test_dates)

    n, t, f = Xtr.shape
    scaler  = StandardScaler().fit(Xtr.reshape(-1, f))
    Xtr     = scaler.transform(Xtr.reshape(-1, f)).reshape(n, t, f)
    Xval    = scaler.transform(Xval.reshape(-1, f)).reshape(Xval.shape[0], t, f)
    Xtest   = scaler.transform(Xtest.reshape(-1, f)).reshape(Xtest.shape[0], t, f)

    model  = LSTMClassifier(n_features=f).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=1e-3)
    lossfn = nn.BCEWithLogitsLoss()

    train_dl = DataLoader(
        TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)),
        batch_size=batch_size, shuffle=True,
    )
    val_t = (torch.tensor(Xval).to(DEVICE), torch.tensor(yval).to(DEVICE))

    best_auc, best_state = -1, None
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = lossfn(model(xb), yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_pred = torch.sigmoid(model(val_t[0])).cpu().numpy()
        auc = roc_auc_score(yval, val_pred)
        if auc > best_auc:
            best_auc, best_state = auc, {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_pred = torch.sigmoid(model(torch.tensor(Xtest).to(DEVICE))).cpu().numpy()

    thresh_lstm = find_best_threshold(yval, val_pred)   # [개선] 임계값 최적화
    result = {
        "AUC":         roc_auc_score(ytest, test_pred),
        "F1":          f1_score(ytest, test_pred >= thresh_lstm, zero_division=0),
        "best_thresh": thresh_lstm,
    }
    return model, scaler, result


# ------------------------------------------------------------------
# 6. SHAP (LightGBM 기준)
# ------------------------------------------------------------------
def shap_summary(model, X_sample, out_png="shap_summary.png"):
    import matplotlib.pyplot as plt

    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    sv = shap_values[1] if isinstance(shap_values, list) else (
        shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values
    )

    shap.summary_plot(sv, X_sample, show=False)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close()
    return explainer, sv


# ------------------------------------------------------------------
# 7. 종목코드 + 날짜로 조회 (LightGBM 기준)
# ------------------------------------------------------------------
def predict_for(model, df, isu_cd, date, threshold=0.5):
    """
    종목코드 + 날짜 → 향후 5거래일 내 공매도 집중(자기 과거 상위 30%) 확률.

    Parameters
    ----------
    threshold : float
        모델 학습 시 얻은 best_thresh 를 넘겨주면 시그널 on/off 도 반환

    Returns
    -------
    dict: ISU_CD, ISU_NM, date, prob_short_concentration_5d,
          signal (prob >= threshold), short_ratio_today,
          short_ratio_pct_vs_own_hist, balance_ratio_today
    """
    isu_cd = str(isu_cd).zfill(6)
    date   = pd.to_datetime(date)

    row = df[(df["ISU_CD"] == isu_cd) & (df["date"] == date)]
    if row.empty:
        raise ValueError(f"데이터 없음: ISU_CD={isu_cd}, date={date.date()}")
    if row[FEATURE_COLS].isna().any(axis=1).iloc[0]:
        raise ValueError(f"피처 결측으로 예측 불가: ISU_CD={isu_cd}, date={date.date()}")

    prob = model.predict_proba(row[FEATURE_COLS])[:, 1][0]
    return {
        "ISU_CD":                      isu_cd,
        "ISU_NM":                      row["ISU_NM"].iloc[0],
        "date":                        date.date(),
        "prob_short_concentration_5d": round(float(prob), 4),
        "signal":                      int(prob >= threshold),
        "short_ratio_today":           row["short_ratio"].iloc[0],
        "short_ratio_pct_vs_own_hist": round(float(row["short_ratio_pct_vs_own_hist"].iloc[0]), 3),
        "balance_ratio_today":         row["balance_ratio"].iloc[0],
    }


def predict_history(model, df, isu_cd, threshold=0.5):
    """특정 종목의 전체 기간 일별 공매도 집중 확률 시계열."""
    isu_cd = str(isu_cd).zfill(6)
    sub    = df[df["ISU_CD"] == isu_cd].dropna(subset=FEATURE_COLS).copy()
    sub["prob_short_concentration_5d"] = model.predict_proba(sub[FEATURE_COLS])[:, 1]
    sub["signal"] = (sub["prob_short_concentration_5d"] >= threshold).astype(int)
    cols = ["ISU_CD", "ISU_NM", "date", "prob_short_concentration_5d",
            "signal", "short_ratio", "balance_ratio", "target"]
    return sub[cols].reset_index(drop=True)


# ------------------------------------------------------------------
# main
# ------------------------------------------------------------------
if __name__ == "__main__":
    df = load_data()
    df = add_features(df)
    df = add_target(df)

    train, val, test = time_split(df)
    print(f"Train: {train['date'].nunique()}일 / "
          f"Val: {val['date'].nunique()}일 / "
          f"Test: {test['date'].nunique()}일")
    print(f"Train target 비율: {train['target'].mean():.3f} / "
          f"Test target 비율: {test['target'].mean():.3f}\n")

    # ---- 테이블/선형 모델 학습 ----
    models, results = train_tabular_models(train, val, test)

    # ---- LSTM ----
    t0 = time.time()
    _, _, lstm_result = train_lstm(df, train, val, test, epochs=10)
    results["LSTM"] = lstm_result
    print(f"(LSTM 학습 시간: {time.time()-t0:.1f}s)\n")

    # ---- 결과 출력 ----
    # 선형 모델과 비선형 모델 구분 출력
    LINEAR_MODELS    = {"Linear Regression", "Logistic Regression"}
    NONLINEAR_MODELS = {"LightGBM", "XGBoost", "CatBoost", "LSTM"}

    print(f"{'Model':>22s} | {'AUC':>6s} | {'F1':>6s} | {'Thresh':>6s} | Type")
    print("-" * 65)
    for name, m in results.items():
        mtype = "선형" if name in LINEAR_MODELS else "비선형"
        print(f"{name:>22s} | {m['AUC']:.4f} | {m['F1']:.4f} | "
              f"{m['best_thresh']:.3f}  | {mtype}")

    # 선형 vs 비선형 F1 차이 요약
    lin_f1s = [results[k]["F1"] for k in LINEAR_MODELS if k in results]
    nl_f1s  = [results[k]["F1"] for k in NONLINEAR_MODELS if k in results]
    if lin_f1s and nl_f1s:
        gap = np.mean(nl_f1s) - np.mean(lin_f1s)
        print(f"\n비선형 평균 F1: {np.mean(nl_f1s):.4f} | "
              f"선형 평균 F1: {np.mean(lin_f1s):.4f} | "
              f"차이: {gap:+.4f} (목표: +0.10 이상)")

    # ---- SHAP ----
    gbm = models["LightGBM"][0]
    shap_summary(gbm, test[FEATURE_COLS].sample(min(500, len(test)), random_state=RANDOM_STATE))
    print("\nSHAP summary plot 저장 완료 → shap_summary.png")

    # ---- 종목 조회 예시 (LightGBM, best_thresh 적용) ----
    latest_date = df["date"].max()
    sample_isu  = df["ISU_CD"].iloc[0]
    gbm_thresh  = results["LightGBM"]["best_thresh"]

    print(f"\n[조회 예시] {sample_isu}, {latest_date.date()} (thresh={gbm_thresh})")
    print(predict_for(gbm, df, sample_isu, latest_date, threshold=gbm_thresh))

    print(f"\n[기업별 시계열 예시] {sample_isu} 최근 5일")
    print(predict_history(gbm, df, sample_isu, threshold=gbm_thresh).tail(5).to_string(index=False))

    # ---- 오늘 기준 전체 종목 위험도 Top 15 ----
    today = df[df["date"] == latest_date].dropna(subset=FEATURE_COLS).copy()
    today["prob"] = gbm.predict_proba(today[FEATURE_COLS])[:, 1]
    top15 = (
        today.sort_values("prob", ascending=False)
             [["ISU_CD", "ISU_NM", "date", "prob"]]
             .head(15)
    )
    print(f"\n오늘({latest_date.date()}) 기준 공매도 주의 종목 Top 15")
    print(top15.to_string(index=False))

Train: 193일 / Val: 20일 / Test: 20일
Train target 비율: 0.370 / Test target 비율: 0.421

(LSTM 학습 시간: 51.3s)

                 Model |    AUC |     F1 | Thresh | Type
-----------------------------------------------------------------
     Linear Regression | 0.8226 | 0.7267 | 0.500  | 선형
   Logistic Regression | 0.8247 | 0.7361 | 0.520  | 선형
              LightGBM | 0.8493 | 0.7639 | 0.410  | 비선형
               XGBoost | 0.8482 | 0.7637 | 0.430  | 비선형
              CatBoost | 0.8433 | 0.7602 | 0.450  | 비선형
                  LSTM | 0.8589 | 0.7632 | 0.400  | 비선형

비선형 평균 F1: 0.7628 | 선형 평균 F1: 0.7314 | 차이: +0.0313 (목표: +0.10 이상)


LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray



SHAP summary plot 저장 완료 → shap_summary.png

[조회 예시] 000020, 2026-03-31 (thresh=0.41)
{'ISU_CD': '000020', 'ISU_NM': '동화약품', 'date': datetime.date(2026, 3, 31), 'prob_short_concentration_5d': 0.086, 'signal': 0, 'short_ratio_today': 2.92, 'short_ratio_pct_vs_own_hist': 0.679, 'balance_ratio_today': 0.08}

[기업별 시계열 예시] 000020 최근 5일
ISU_CD ISU_NM       date  prob_short_concentration_5d  signal  short_ratio  balance_ratio  target
000020   동화약품 2026-03-25                     0.026135       0         0.24           0.09       0
000020   동화약품 2026-03-26                     0.008843       0         0.17           0.08       0
000020   동화약품 2026-03-27                     0.011462       0         1.14           0.08       0
000020   동화약품 2026-03-30                     0.039850       0         3.14           0.09       0
000020   동화약품 2026-03-31                     0.086023       0         2.92           0.08       0

오늘(2026-03-31) 기준 공매도 주의 종목 Top 15
ISU_CD   ISU_NM       date     prob
000640 

In [4]:
!pip install dice_ml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 9.8 MB/s eta 0:00:000m eta 0:00:0136m0:00:01m
